In [1]:
!pip install beautifulsoup4 selenium pandas

In [5]:
# 필요한 모듈 임포트
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import time
from bs4 import BeautifulSoup
import pandas as pd
import random
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")  # Overcome limited resource problems
driver = webdriver.Chrome(options=chrome_options)

# CSV 파일 불러오기
df_companies = pd.read_csv('상장법인목록_2024_08_09.csv', encoding='utf-8')

# '회사명' 열만 추출
company_names = df_companies['회사명'].tolist()

# 크롬드라이버 실행
driver = webdriver.Chrome()

# 크롬 드라이버에 URL 주소 넣고 실행
driver.get('https://insight.wanted.co.kr/')
time.sleep(3)

# xpath를 사용하여 요소를 가져옴
element = driver.find_element(By.XPATH, '//*[@id="__next"]/div[1]/nav/div/div[2]/a')
element.click()
time.sleep(random.uniform(1, 3)) 

# 전체 데이터를 저장할 리스트 생성
data_list = []

# 첫번째 회사 검색 및 처리
search_box = driver.find_element(By.XPATH, '//*[@id="__next"]/div[2]/div/div/form/div/div/input')
search_box.send_keys(company_names[0])
search_box.send_keys(Keys.RETURN)
time.sleep(random.uniform(1, 3)) 

company_link = driver.find_element(By.XPATH, '//*[@id="__next"]/div[2]/div/div/form/div[2]/ul/li/a')
href = company_link.get_attribute('href')

# 링크로 이동
driver.get(href)
time.sleep(random.uniform(1, 3)) 

# 데이터가 들어있는 부분을 BeautifulSoup으로 가져오기
soup = BeautifulSoup(driver.page_source, 'html.parser')

# 첫 번째 회사의 데이터를 추출하여 리스트에 추가
data = {
    "회사": company_names[0],
    "표준산업분류": soup.find('dt', string='표준산업분류').find_next('dd').text if soup.find('dt', string='표준산업분류') else '',
    "연혁": soup.find('dt', string='연혁').find_next('dd').text if soup.find('dt', string='연혁') else '',
    "매출액": soup.find('dt', string='매출액').find_next('dd').text if soup.find('dt', string='매출액') else '',
    "기업유형": soup.find('dt', string='기업유형').find_next('dd').text if soup.find('dt', string='기업유형') else '',
    "평균연봉": soup.find('dt', string='평균연봉').find_next('dd').text if soup.find('dt', string='평균연봉') else '',
    "홈페이지": soup.find('dt', string='홈페이지').find_next('dd').text if soup.find('dt', string='홈페이지') else '',
    "고용보험 사업장 수": soup.find('dt', string='고용보험 사업장 수').find_next('dd').text if soup.find('dt', string='고용보험 사업장 수') else '',
    "고용보험 가입 사원수": soup.find('dt', string='고용보험 가입 사원수').find_next('dd').text if soup.find('dt', string='고용보험 가입 사원수') else '',
    "국민연금 가입 사원수": soup.find('dt', string='국민연금 가입 사원수').find_next('dd').text if soup.find('dt', string='국민연금 가입 사원수') else '',
    "퇴사/입사 (1년)": soup.find('dt', string='퇴사/입사 (1년)').find_next('dd').text if soup.find('dt', string='퇴사/입사 (1년)') else ''
}

# 리스트에 첫 번째 데이터 추가
data_list.append(data)

# 나머지 회사 검색 및 처리
for company_name in company_names[1:]:
    time.sleep(1)
    
    try:
        # 새 검색 버튼 클릭
        new_search = driver.find_element(By.XPATH, '//*[@id="__next"]/div[1]/div[2]/nav/aside/ul/li[1]/button')
        new_search.click()
        time.sleep(random.uniform(1, 3)) 
        
        # 회사명 입력 및 검색 실행
        search_input = driver.find_element(By.XPATH, '//*[@id="nav_searchbar"]/div/div[2]/div/form/input')
        search_input.send_keys(company_name)
        search_input.send_keys(Keys.RETURN)
        time.sleep(random.uniform(1, 3)) 

        # 검색 결과 링크 클릭
        company_link = driver.find_element(By.XPATH, '//*[@id="search_tabpanel_overview"]/div/div[2]/ul/li/a')
        href = company_link.get_attribute('href')

        # 페이지 이동
        driver.get(href)
        time.sleep(random.uniform(1, 3)) 

        # 페이지에서 데이터 추출
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        time.sleep(random.uniform(1, 3)) 
        data = {
            "회사": company_name,
            "표준산업분류": soup.find('dt', string='표준산업분류').find_next('dd').text if soup.find('dt', string='표준산업분류') else '',
            "연혁": soup.find('dt', string='연혁').find_next('dd').text if soup.find('dt', string='연혁') else '',
            "매출액": soup.find('dt', string='매출액').find_next('dd').text if soup.find('dt', string='매출액') else '',
            "기업유형": soup.find('dt', string='기업유형').find_next('dd').text if soup.find('dt', string='기업유형') else '',
            "평균연봉": soup.find('dt', string='평균연봉').find_next('dd').text if soup.find('dt', string='평균연봉') else '',
            "홈페이지": soup.find('dt', string='홈페이지').find_next('dd').text if soup.find('dt', string='홈페이지') else '',
            "고용보험 사업장 수": soup.find('dt', string='고용보험 사업장 수').find_next('dd').text if soup.find('dt', string='고용보험 사업장 수') else '',
            "고용보험 가입 사원수": soup.find('dt', string='고용보험 가입 사원수').find_next('dd').text if soup.find('dt', string='고용보험 가입 사원수') else '',
            "국민연금 가입 사원수": soup.find('dt', string='국민연금 가입 사원수').find_next('dd').text if soup.find('dt', string='국민연금 가입 사원수') else '',
            "퇴사/입사 (1년)": soup.find('dt', string='퇴사/입사 (1년)').find_next('dd').text if soup.find('dt', string='퇴사/입사 (1년)') else ''
        }
        

        # 리스트에 데이터 추가
        data_list.append(data)
        
    except Exception:
        print(f"No results found for {company_name}, skipping.")
        continue  # 검색 결과가 없으면 다음으로 넘어감

# 리스트를 데이터프레임으로 변환
df_total = pd.DataFrame(data_list)

# 최종 데이터프레임을 CSV 파일로 저장
df_total.to_csv('new_company_data.csv', encoding='utf-8-sig', index=False)

# 크롬 드라이버 종료
driver.quit()


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=130.0.6723.92)
Stacktrace:
0   chromedriver                        0x00000001013c7648 cxxbridge1$str$ptr + 3645404
1   chromedriver                        0x00000001013bfea8 cxxbridge1$str$ptr + 3614780
2   chromedriver                        0x0000000100e2c104 cxxbridge1$string$len + 88416
3   chromedriver                        0x0000000100e07b70 core::str::slice_error_fail::h1cab30ac4b13c655 + 3792
4   chromedriver                        0x0000000100e9466c cxxbridge1$string$len + 515784
5   chromedriver                        0x0000000100ea7638 cxxbridge1$string$len + 593556
6   chromedriver                        0x0000000100e62f54 cxxbridge1$string$len + 313264
7   chromedriver                        0x0000000100e63ba4 cxxbridge1$string$len + 316416
8   chromedriver                        0x00000001013921e8 cxxbridge1$str$ptr + 3427196
9   chr

In [ ]:
sudo yum install -y google-chrome-stable
wget https://chromedriver.storage.googleapis.com/114.0.5735.90/chromedriver_linux64.zip
unzip chromedriver_linux64.zip
sudo mv chromedriver /usr/local/bin/

In [2]:
# 필요한 모듈 임포트
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import time
from bs4 import BeautifulSoup
import pandas as pd
import random
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")  # Overcome limited resource problems
driver = webdriver.Chrome(options=chrome_options)

# CSV 파일 불러오기
df_companies = pd.read_csv('상장법인목록_2024_08_09.csv', encoding='utf-8')
df_previous = pd.read_csv('company.csv', encoding='utf-8')

prev_company_names = df_previous['회사'].tolist()

# '회사명' 열만 추출
company_names = df_companies['회사명'].tolist()

# Remove company names that are already in prev_company_names
filtered_company_names = [name for name in company_names if name not in prev_company_names]

# 크롬드라이버 실행
driver = webdriver.Chrome()

# 크롬 드라이버에 URL 주소 넣고 실행
driver.get('https://insight.wanted.co.kr/')
time.sleep(3)


# xpath를 사용하여 요소를 가져옴
element = driver.find_element(By.XPATH, '//*[@id="__next"]/div[1]/nav/div/div[2]/a')
element.click()
time.sleep(random.uniform(1, 3)) 

# 전체 데이터를 저장할 리스트 생성
data_list = []

# 첫번째 회사 검색 및 처리
search_box = driver.find_element(By.XPATH, '//*[@id="__next"]/div[2]/div/div/form/div/div/input')
search_box.send_keys(company_names[0])
search_box.send_keys(Keys.RETURN)
time.sleep(random.uniform(1, 3)) 

company_link = driver.find_element(By.XPATH, '//*[@id="__next"]/div[2]/div/div/form/div[2]/ul/li/a')
href = company_link.get_attribute('href')

# 링크로 이동
driver.get(href)
time.sleep(random.uniform(1, 3)) 

# 데이터가 들어있는 부분을 BeautifulSoup으로 가져오기
soup = BeautifulSoup(driver.page_source, 'html.parser')

# 첫 번째 회사의 데이터를 추출하여 리스트에 추가
data = {
    "회사": filtered_company_names[0],
    "표준산업분류": soup.find('dt', string='표준산업분류').find_next('dd').text if soup.find('dt', string='표준산업분류') else '',
    "연혁": soup.find('dt', string='연혁').find_next('dd').text if soup.find('dt', string='연혁') else '',
    "매출액": soup.find('dt', string='매출액').find_next('dd').text if soup.find('dt', string='매출액') else '',
    "기업유형": soup.find('dt', string='기업유형').find_next('dd').text if soup.find('dt', string='기업유형') else '',
    "평균연봉": soup.find('dt', string='평균연봉').find_next('dd').text if soup.find('dt', string='평균연봉') else '',
    "홈페이지": soup.find('dt', string='홈페이지').find_next('dd').text if soup.find('dt', string='홈페이지') else '',
    "고용보험 사업장 수": soup.find('dt', string='고용보험 사업장 수').find_next('dd').text if soup.find('dt', string='고용보험 사업장 수') else '',
    "고용보험 가입 사원수": soup.find('dt', string='고용보험 가입 사원수').find_next('dd').text if soup.find('dt', string='고용보험 가입 사원수') else '',
    "국민연금 가입 사원수": soup.find('dt', string='국민연금 가입 사원수').find_next('dd').text if soup.find('dt', string='국민연금 가입 사원수') else '',
    "퇴사/입사 (1년)": soup.find('dt', string='퇴사/입사 (1년)').find_next('dd').text if soup.find('dt', string='퇴사/입사 (1년)') else ''
}

# 리스트에 첫 번째 데이터 추가
data_list.append(data)



# Now use the filtered list for web scraping
for company_name in filtered_company_names[1:]:
    time.sleep(1)
    
    try:
        # Same scraping process as before
        new_search = driver.find_element(By.XPATH, '//*[@id="__next"]/div[1]/div[2]/nav/aside/ul/li[1]/button')
        new_search.click()
        time.sleep(random.uniform(1, 3)) 
        
        # Company name search and scrape process
        search_input = driver.find_element(By.XPATH, '//*[@id="nav_searchbar"]/div/div[2]/div/form/input')
        search_input.send_keys(company_name)
        search_input.send_keys(Keys.RETURN)
        time.sleep(random.uniform(1, 3)) 

        company_link = driver.find_element(By.XPATH, '//*[@id="search_tabpanel_overview"]/div/div[2]/ul/li/a')
        href = company_link.get_attribute('href')

        # Continue the scraping process
        driver.get(href)
        time.sleep(random.uniform(1, 3)) 

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        data = {
            "회사": company_name,
            "표준산업분류": soup.find('dt', string='표준산업분류').find_next('dd').text if soup.find('dt', string='표준산업분류') else '',
            "연혁": soup.find('dt', string='연혁').find_next('dd').text if soup.find('dt', string='연혁') else '',
            "매출액": soup.find('dt', string='매출액').find_next('dd').text if soup.find('dt', string='매출액') else '',
            "기업유형": soup.find('dt', string='기업유형').find_next('dd').text if soup.find('dt', string='기업유형') else '',
            "평균연봉": soup.find('dt', string='평균연봉').find_next('dd').text if soup.find('dt', string='평균연봉') else '',
            "홈페이지": soup.find('dt', string='홈페이지').find_next('dd').text if soup.find('dt', string='홈페이지') else '',
            "고용보험 사업장 수": soup.find('dt', string='고용보험 사업장 수').find_next('dd').text if soup.find('dt', string='고용보험 사업장 수') else '',
            "고용보험 가입 사원수": soup.find('dt', string='고용보험 가입 사원수').find_next('dd').text if soup.find('dt', string='고용보험 가입 사원수') else '',
            "국민연금 가입 사원수": soup.find('dt', string='국민연금 가입 사원수').find_next('dd').text if soup.find('dt', string='국민연금 가입 사원수') else '',
            "퇴사/입사 (1년)": soup.find('dt', string='퇴사/입사 (1년)').find_next('dd').text if soup.find('dt', string='퇴사/입사 (1년)') else ''
        }
        
        data_list.append(data)
        
    except Exception:
        print(f"No results found for {company_name}, skipping.")
        continue  # 검색 결과가 없으면 다음으로 넘어감

# 리스트를 데이터프레임으로 변환
df_total = pd.DataFrame(data_list)

# 최종 데이터프레임을 CSV 파일로 저장
df_total.to_csv('new_company_data.csv', encoding='utf-8-sig', index=False)

# 크롬 드라이버 종료
driver.quit()


No results found for 엔에이치스팩31호, skipping.
No results found for SK증권제13호스팩, skipping.
No results found for 이베스트스팩6호, skipping.
No results found for 신한글로벌액티브리츠, skipping.
No results found for 한국제15호스팩, skipping.
No results found for 미래에셋비전스팩6호, skipping.
No results found for 에이치엠씨제7호스팩, skipping.
No results found for KB제29호스팩, skipping.
No results found for 미래에셋비전스팩5호, skipping.
No results found for 한국제14호스팩, skipping.
No results found for 디비금융스팩12호, skipping.
No results found for 미래에셋비전스팩4호, skipping.
No results found for KB제28호스팩, skipping.
No results found for SK증권제12호스팩, skipping.
No results found for 유안타제16호스팩, skipping.
No results found for 하나33호스팩, skipping.
No results found for 신한제13호스팩, skipping.
No results found for 신한제12호스팩, skipping.
No results found for SK이터닉스, skipping.
No results found for 하나32호스팩, skipping.
No results found for 비엔케이제2호스팩, skipping.
No results found for 하나31호스팩, skipping.
No results found for SK증권제11호스팩, skipping.
No results found for 유안타제15호스팩, skipping.


In [1]:
import pandas as pd

# CSV 파일 불러오기
df_companies = pd.read_csv('company.csv', encoding='utf-8')
df_newcompanies = pd.read_csv('new_company_data.csv', encoding='utf-8')

# 두 DataFrame을 행 방향으로 결합
df_combined = pd.concat([df_companies, df_newcompanies], ignore_index=True)

# 결과 확인
print(df_combined.head())

# 병합된 데이터프레임을 새 CSV 파일로 저장
df_combined.to_csv('company_data.csv', encoding='utf-8-sig', index=False)


         회사                 표준산업분류              연혁           매출액   기업유형  \
0  아이빔테크놀로지        사진장비 및 광학기기 제조업   7년 (2017년 설립)             -   주식회사   
1  피앤에스미캐닉스             엔지니어링 서비스업  26년 (1998년 설립)             -  기타 법인   
2      산일전기            건물설비 설치 공사업  26년 (1998년 설립)  648억 3,274만원  기타 법인   
3   엑셀세라퓨틱스             자연과학 연구개발업   9년 (2015년 설립)   19억 6,628만원   주식회사   
4      시프트업  시스템ㆍ응용 소프트웨어 개발 및 공급업  11년 (2013년 설립)           비공개   주식회사   

      평균연봉                       홈페이지 고용보험 사업장 수 고용보험 가입 사원수 국민연금 가입 사원수  \
0  5,055만원                          -         1개         26명         36명   
1  5,043만원                          -         1개         18명         22명   
2  5,587만원                          -         2개        220명        244명   
3  4,430만원            www.xcell.media         1개         55명         55명   
4      비공개  https://www.shiftup.co.kr         1개         비공개         비공개   

  퇴사/입사 (1년)  
0      4명/8명  
1     11명/8명  
2    41명/89명  
3    30명/29명  
4        비공개  
